# Open Insurance Demo — Centralized MCP with Auth Propagation

This notebook demonstrates **centralized MCP server management** with **Okta OIDC auth propagation** using AWS AgentCore Gateway — themed for an **insurance** use case.

## The "whoa" moments

1. **One Gateway endpoint** manages both Lambda functions AND MCP servers
2. **User identity flows through** — each user's JWT is validated by the Gateway
3. **Different users, different tools** — Alice (customer service) sees only policy lookup; Bob (claims adjuster) sees both
4. **Gateway-level ENFORCE** — Cedar policies block unauthorized tool calls at the Gateway, not the agent

## Insurance Demo Personas

| User | Role | Tools Available |
|------|------|----------------|
| Alice | Customer Service Rep | `PolicyLookup___lookup_policy` |
| Bob | Claims Adjuster | `PolicyLookup___lookup_policy` + `ClaimsData___query_claims` |

## Prerequisites

- `00_okta_setup.ipynb` completed (auth server scopes, claims, access policy)
- `01_agentcore_setup.ipynb` completed (Gateway, Lambda targets, users, Cedar policies)
- `gateway_config.json` exists with deployed resource IDs

## What this notebook does

| Cells | What | Why |
|-------|------|-----|
| 1-2 | Create Okta Web App (ROPC) | Confidential client for programmatic token exchange |
| 3 | Enable ROPC grant + password-only auth policy | OIE defaults block password grant |
| 4 | Assign demo users to Web App | Alice + Bob (from 01_agentcore_setup) |
| 5 | Register client with Gateway | Add Web App client ID to `allowedClients` |
| 6 | Verify token | Confirm all JWT claims are correct |
| 7+ | Run demo | Pre-auth users, create agents, test access control |

## Cell 1: Create Okta Web Application (ROPC)

Creates an OIDC **Web Application** (confidential client) via the Okta Management API. This is used for the **Resource Owner Password** grant — a server-side flow where the agent exchanges user credentials for a JWT programmatically (no browser popup).

The `CLIENT_ID` and `CLIENT_SECRET` are written to `.env` for use in subsequent cells.

In [ ]:
import os
import re
import json
import urllib.parse
import base64
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

OKTA_DOMAIN = os.environ["OKTA_DOMAIN"]
OKTA_API_TOKEN = os.environ["OKTA_API_TOKEN"]
OKTA_ISSUER = f"https://{OKTA_DOMAIN}/oauth2/default"

HEADERS = {
    "Authorization": f"SSWS {OKTA_API_TOKEN}",
    "Content-Type": "application/json",
}

AUTH_SERVER_BASE = f"https://{OKTA_DOMAIN}/api/v1/authorizationServers/default"


def set_env_var(content, key, value):
    """Set or update a key=value pair in .env file content."""
    pattern = re.compile(rf"^{re.escape(key)}=.*$", re.MULTILINE)
    if pattern.search(content):
        return pattern.sub(f"{key}={value}", content)
    else:
        return content.rstrip() + f"\n{key}={value}\n"


# --- Create OIDC Web Application ---
APP_LABEL = "AgentCore Gateway Demo (Web)"

existing_apps = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/apps?q={urllib.parse.quote(APP_LABEL)}&limit=10",
    headers=HEADERS,
)
existing_app = next(
    (a for a in existing_apps.json() if a.get("label") == APP_LABEL), None
)

if existing_app:
    app = existing_app
    OKTA_CLIENT_ID = app["credentials"]["oauthClient"]["client_id"]
    print(f"App '{APP_LABEL}' already exists")
    print(f"  Client ID: {OKTA_CLIENT_ID}")
    OKTA_CLIENT_SECRET = os.environ.get("OKTA_CLIENT_SECRET", "")
    if not OKTA_CLIENT_SECRET:
        print(f"\n  OKTA_CLIENT_SECRET not found in .env — add it manually or delete the app and re-run.")
else:
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/apps",
        headers=HEADERS,
        json={
            "name": "oidc_client",
            "label": APP_LABEL,
            "signOnMode": "OPENID_CONNECT",
            "settings": {
                "oauthClient": {
                    "application_type": "web",
                    "grant_types": ["authorization_code", "refresh_token"],
                    "response_types": ["code"],
                    "redirect_uris": ["http://localhost:8080/authorization-code/callback"],
                    "post_logout_redirect_uris": ["http://localhost:8080"],
                }
            },
            "credentials": {
                "oauthClient": {
                    "token_endpoint_auth_method": "client_secret_basic",
                }
            },
        },
    )
    resp.raise_for_status()
    app = resp.json()
    OKTA_CLIENT_ID = app["credentials"]["oauthClient"]["client_id"]
    OKTA_CLIENT_SECRET = app["credentials"]["oauthClient"]["client_secret"]
    print(f"Created app '{APP_LABEL}'")
    print(f"  Client ID:     {OKTA_CLIENT_ID}")
    print(f"  Client Secret: {OKTA_CLIENT_SECRET[:8]}...  (saved to .env)")

# Assign app to Everyone group
everyone_resp = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/groups?q=Everyone&limit=1",
    headers=HEADERS,
)
everyone_group_id = everyone_resp.json()[0]["id"]
assign_resp = requests.put(
    f"https://{OKTA_DOMAIN}/api/v1/apps/{app['id']}/groups/{everyone_group_id}",
    headers=HEADERS,
)
if assign_resp.status_code in (200, 204):
    print(f"  Assigned to Everyone group")

APP_ID = app["id"]

# Write CLIENT_ID and CLIENT_SECRET to .env
env_path = os.path.join(os.getcwd(), ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        env_content = f.read()
else:
    env_content = ""

env_content = set_env_var(env_content, "OKTA_CLIENT_ID", OKTA_CLIENT_ID)
if OKTA_CLIENT_SECRET:
    env_content = set_env_var(env_content, "OKTA_CLIENT_SECRET", OKTA_CLIENT_SECRET)

with open(env_path, "w") as f:
    f.write(env_content)

print(f"\n  OKTA_CLIENT_ID and OKTA_CLIENT_SECRET written to .env")

## Cell 2: Enable ROPC Grant + Password-Only Auth Policy

Okta OIE trials don't expose the ROPC grant type in the UI, and default to MFA which blocks password-only auth.
We enable the `password` grant via the Management API and create a 1FA policy assigned to the app.

In [ ]:
# --- Enable ROPC grant type on the Web App ---
resp = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/apps/{APP_ID}",
    headers=HEADERS,
)
resp.raise_for_status()
app_config = resp.json()

current_grants = app_config["settings"]["oauthClient"]["grant_types"]
print(f"Current grant_types: {current_grants}")

required_grants = ["authorization_code", "refresh_token", "password", "client_credentials"]
if set(required_grants).issubset(set(current_grants)):
    print("All required grant types already enabled")
else:
    app_config["settings"]["oauthClient"]["grant_types"] = required_grants
    resp = requests.put(
        f"https://{OKTA_DOMAIN}/api/v1/apps/{APP_ID}",
        headers=HEADERS,
        json=app_config,
    )
    resp.raise_for_status()
    updated_grants = resp.json()["settings"]["oauthClient"]["grant_types"]
    print(f"Updated grant_types: {updated_grants}")

print("✅ Resource Owner Password grant enabled")

# --- Create password-only auth policy ---
AUTH_POLICY_NAME = "Password Only (Demo)"

existing = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/policies?type=ACCESS_POLICY",
    headers=HEADERS,
)
existing_policy = next(
    (p for p in existing.json() if p["name"] == AUTH_POLICY_NAME), None
)

if existing_policy:
    AUTH_POLICY_ID = existing_policy["id"]
    print(f"\nAuth policy '{AUTH_POLICY_NAME}' already exists (id: {AUTH_POLICY_ID})")
else:
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/policies",
        headers=HEADERS,
        json={
            "name": AUTH_POLICY_NAME,
            "description": "1FA password-only policy for ROPC demo flow",
            "type": "ACCESS_POLICY",
            "status": "ACTIVE",
        },
    )
    resp.raise_for_status()
    AUTH_POLICY_ID = resp.json()["id"]
    print(f"\nCreated auth policy (id: {AUTH_POLICY_ID})")

# Create 1FA rule
AUTH_RULE_NAME = "Password only"
existing_rules = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/policies/{AUTH_POLICY_ID}/rules",
    headers=HEADERS,
)
existing_rule = next(
    (r for r in existing_rules.json() if r["name"] == AUTH_RULE_NAME), None
)

if existing_rule:
    print(f"Rule '{AUTH_RULE_NAME}' already exists")
else:
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/policies/{AUTH_POLICY_ID}/rules",
        headers=HEADERS,
        json={
            "name": AUTH_RULE_NAME,
            "type": "ACCESS_POLICY",
            "status": "ACTIVE",
            "priority": 1,
            "conditions": {
                "network": {"connection": "ANYWHERE"}
            },
            "actions": {
                "appSignOn": {
                    "access": "ALLOW",
                    "verificationMethod": {
                        "factorMode": "1FA",
                        "type": "ASSURANCE",
                        "reauthenticateIn": "PT43800H",
                        "constraints": [
                            {"knowledge": {"types": ["password"]}}
                        ],
                    },
                }
            },
        },
    )
    resp.raise_for_status()
    print(f"Created 1FA rule")

# Assign policy to the app
resp = requests.put(
    f"https://{OKTA_DOMAIN}/api/v1/apps/{APP_ID}/policies/{AUTH_POLICY_ID}",
    headers=HEADERS,
)
if resp.status_code in (200, 204):
    print(f"Assigned password-only policy to app")
else:
    print(f"Warning: assign policy returned {resp.status_code} — {resp.text[:200]}")

print("\n✅ ROPC grant and password-only auth policy configured")

## Cell 3: Assign Demo Users to Web App

Assigns Alice and Bob (created by `01_agentcore_setup.ipynb`) to this Web Application so they can authenticate via ROPC grant.

In [ ]:
# --- Load user credentials from .env (written by 01_agentcore_setup) ---
load_dotenv(override=True)

ALICE_USERNAME = os.environ["ALICE_USERNAME"]
ALICE_PASSWORD = os.environ["ALICE_PASSWORD"]
BOB_USERNAME = os.environ["BOB_USERNAME"]
BOB_PASSWORD = os.environ["BOB_PASSWORD"]

# --- Look up existing users and assign to this Web App ---
for display_name, username in [("Alice", ALICE_USERNAME), ("Bob", BOB_USERNAME)]:
    search = requests.get(
        f"https://{OKTA_DOMAIN}/api/v1/users?q={username}&limit=1",
        headers=HEADERS,
    )
    users = search.json()
    match = next((u for u in users if u["profile"]["login"] == username), None)
    if match:
        resp = requests.put(
            f"https://{OKTA_DOMAIN}/api/v1/apps/{APP_ID}/users/{match['id']}",
            headers=HEADERS,
            json={"id": match["id"], "scope": "USER"},
        )
        if resp.status_code in (200, 409):
            print(f"{display_name} ({username}) assigned to Web App")
        else:
            print(f"Warning: assign {display_name} returned {resp.status_code}")
    else:
        print(f"Error: user '{username}' not found — run 01_agentcore_setup.ipynb first")

print(f"\n--- Demo Users ---")
print(f"Alice: {ALICE_USERNAME}")
print(f"Bob:   {BOB_USERNAME}")
print(f"\n✅ Users assigned to Web App")

## Cell 4: Register Web App Client with Gateway

The AgentCore Gateway validates the JWT `client_id` claim against its `allowedClients` list.
We add the Web App's client ID so tokens issued by this app are accepted.

In [ ]:
import time
import boto3

with open("gateway_config.json") as f:
    config = json.load(f)

GATEWAY_ID = config["gateway_id"]
AWS_REGION = config["aws_region"]

agentcore = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

# --- Get current Gateway config ---
gw = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
current_clients = gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedClients"]
print(f"Current allowedClients: {current_clients}")

# --- Add Web App client ID if not already present ---
if OKTA_CLIENT_ID in current_clients:
    print(f"\nWeb App client ID already in allowedClients — no update needed")
else:
    new_clients = current_clients + [OKTA_CLIENT_ID]
    response = agentcore.update_gateway(
        gatewayIdentifier=GATEWAY_ID,
        name=gw["name"],
        roleArn=gw["roleArn"],
        protocolType=gw["protocolType"],
        authorizerType=gw["authorizerType"],
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": gw["authorizerConfiguration"]["customJWTAuthorizer"]["discoveryUrl"],
                "allowedAudience": gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedAudience"],
                "allowedClients": new_clients,
            }
        },
        policyEngineConfiguration=gw.get("policyEngineConfiguration", {}),
        exceptionLevel="DEBUG",
    )
    print(f"Updated allowedClients: {new_clients}")
    print(f"Gateway status: {response['status']}")

    # Wait for READY
    for attempt in range(12):
        gw = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
        if gw["status"] == "READY":
            print("Gateway READY")
            break
        time.sleep(5)
        print(f"  Status: {gw['status']} ({(attempt + 1) * 5}s)")

## Cell 5: Verify Token Configuration

Request an access token for Alice using the ROPC grant and verify all expected claims are present.

Expected claims: `aud`, `iss`, `client_id`, `groups` (with `analysts`), `sub`.

In [ ]:
# Request token for Alice
token_resp = requests.post(
    f"{OKTA_ISSUER}/v1/token",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={
        "grant_type": "password",
        "username": ALICE_USERNAME,
        "password": ALICE_PASSWORD,
        "scope": "openid profile groups",
        "client_id": OKTA_CLIENT_ID,
        "client_secret": OKTA_CLIENT_SECRET,
    },
)

if token_resp.status_code != 200:
    print(f"Token request failed: {token_resp.json()}")
    err = token_resp.json().get("error", "")
    if "unauthorized_client" in err:
        print("  -> password grant not enabled. Re-run Cell 2.")
    elif "invalid_scope" in err:
        print("  -> 'groups' scope missing. Re-run 00_okta_setup Cell 2.")
    elif "access_denied" in err:
        print("  -> No access policy allows this grant. Re-run 00_okta_setup Cell 3.")
    elif "sign on policy" in str(token_resp.json()):
        print("  -> Auth policy requires MFA. Re-run Cell 2.")
    raise SystemExit("Fix the error above and re-run this cell")

access_token = token_resp.json()["access_token"]

# Decode JWT payload (base64url -> JSON)
payload_b64 = access_token.split(".")[1]
padding = 4 - len(payload_b64) % 4
if padding < 4:
    payload_b64 += "=" * padding
payload = json.loads(base64.urlsafe_b64decode(payload_b64))

print("Access token claims:")
print(json.dumps(payload, indent=2))

# Verify expected claims
checks = [
    ("aud" in payload, f"aud: {payload.get('aud')}"),
    ("iss" in payload, f"iss: {payload.get('iss')}"),
    ("client_id" in payload, f"client_id: {payload.get('client_id')}"),
    ("groups" in payload, f"groups: {payload.get('groups')}"),
    ("sub" in payload, f"sub: {payload.get('sub')}"),
    ("analysts" in payload.get("groups", []), "Alice is in 'analysts' group"),
]

print("\n--- Claim Verification ---")
all_pass = True
for passed, label in checks:
    icon = "PASS" if passed else "FAIL"
    print(f"  [{icon}] {label}")
    if not passed:
        all_pass = False

if all_pass:
    print("\n✅ All claims verified — ready to run the demo!")
else:
    print("\nSome claims are missing — check the cells above.")

## Cell 6: Load Config + Pre-Authenticate Both Users

We authenticate both Alice and Bob upfront via Okta Resource Owner Password flow, then create separate Strands agents for each — one connected to the Gateway with Alice's token, one with Bob's.

The Gateway validates each JWT against Okta's JWKS endpoint and checks:
- **Audience** matches `api://default`
- **Client ID** matches the configured `allowedClients`
- **Signature** is valid per Okta's signing keys

Cedar policies in **ENFORCE mode** then control which tools each user can discover and invoke.

In [ ]:
import jwt
import httpx
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamable_http_client

# --- Load Gateway config (URL already loaded in Cell 4) ---
GATEWAY_URL = config["gateway_url"]

# --- Model to use for Strands agents ---
MODEL_ID = "au.anthropic.claude-sonnet-4-6"


def get_okta_token(username: str, password: str) -> dict:
    """Authenticate a user via Okta Resource Owner Password flow and return the JWT."""
    token_url = f"{OKTA_ISSUER}/v1/token"
    response = requests.post(
        token_url,
        data={
            "grant_type": "password",
            "username": username,
            "password": password,
            "scope": "openid profile groups",
            "client_id": OKTA_CLIENT_ID,
            "client_secret": OKTA_CLIENT_SECRET,
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    response.raise_for_status()
    token_data = response.json()
    access_token = token_data["access_token"]

    # Decode JWT (without verification, just to display claims)
    claims = jwt.decode(access_token, options={"verify_signature": False})
    return {"access_token": access_token, "claims": claims}


# --- Pre-authenticate both users ---
print("Authenticating Alice (customer service rep)...")
alice_auth = get_okta_token(ALICE_USERNAME, ALICE_PASSWORD)
print(f"  User: {alice_auth['claims'].get('sub', 'unknown')}")
print(f"  Groups: {alice_auth['claims'].get('groups', [])}")

print("\nAuthenticating Bob (claims adjuster)...")
bob_auth = get_okta_token(BOB_USERNAME, BOB_PASSWORD)
print(f"  User: {bob_auth['claims'].get('sub', 'unknown')}")
print(f"  Groups: {bob_auth['claims'].get('groups', [])}")

# --- Create MCPClients connected to Gateway with each user's token ---
alice_http = httpx.AsyncClient(headers={"Authorization": f"Bearer {alice_auth['access_token']}"})
bob_http = httpx.AsyncClient(headers={"Authorization": f"Bearer {bob_auth['access_token']}"})

alice_client = MCPClient(
    lambda: streamable_http_client(GATEWAY_URL, http_client=alice_http),
    startup_timeout=60,
)

bob_client = MCPClient(
    lambda: streamable_http_client(GATEWAY_URL, http_client=bob_http),
    startup_timeout=60,
)

# --- Create Strands Agents ---
alice_client.__enter__()
bob_client.__enter__()

alice_tools = alice_client.list_tools_sync()
bob_tools = bob_client.list_tools_sync()

SYSTEM_PROMPT = """You are a helpful assistant for an insurance company.
If a tool call fails with an access denied or policy enforcement error, explain clearly
that the user does not have permission to use that tool."""

alice_agent = Agent(
    model=MODEL_ID,
    tools=alice_tools,
    system_prompt=SYSTEM_PROMPT,
)

bob_agent = Agent(
    model=MODEL_ID,
    tools=bob_tools,
    system_prompt=SYSTEM_PROMPT,
)

alice_tool_names = [t.tool_name if hasattr(t, 'tool_name') else str(t) for t in alice_tools]
bob_tool_names = [t.tool_name if hasattr(t, 'tool_name') else str(t) for t in bob_tools]

print(f"\n--- Gateway Authentication (ENFORCE mode) ---")
print(f"  Gateway URL: {GATEWAY_URL}")
print(f"  Model:       {MODEL_ID}")
print(f"  Alice tools: {alice_tool_names}")
print(f"  Bob tools:   {bob_tool_names}")
print(f"\n  Alice sees {len(alice_tools)} tool(s), Bob sees {len(bob_tools)} tool(s)")
print(f"  Access control enforced by Cedar policies at the Gateway")

## Cell 7: Show Available Tools Per User

With ENFORCE mode, the Gateway filters tool discovery based on Cedar policies.
Alice (customer service) only sees policy lookup tools; Bob (claims adjuster) sees both policy and claims tools.

In [2]:
print("Tools visible through AgentCore Gateway:\n")

print("ALICE (customer service rep):")
for tool in alice_tools:
    name = tool.tool_name if hasattr(tool, 'tool_name') else str(tool)
    print(f"  - {name}")
print(f"  Total: {len(alice_tools)} tool(s)\n")

print("BOB (claims adjuster):")
for tool in bob_tools:
    name = tool.tool_name if hasattr(tool, 'tool_name') else str(tool)
    print(f"  - {name}")
print(f"  Total: {len(bob_tools)} tool(s)\n")

print("Cedar ENFORCE mode filters tool discovery per user.")
print("Alice cannot even see ClaimsData — the Gateway hides it from her.")

Tools visible through AgentCore Gateway:

ALICE (customer service rep):
  - PolicyLookup___lookup_policy
  Total: 1 tool(s)

BOB (claims adjuster):
  - ClaimsData___query_claims
  - PolicyLookup___lookup_policy
  Total: 2 tool(s)

Cedar ENFORCE mode filters tool discovery per user.
Alice cannot even see ClaimsData — the Gateway hides it from her.


## Cell 8: Alice Looks Up a Policy (ALLOWED)

Alice is a customer service representative. The Cedar policy allows **all authenticated users** to access `PolicyLookup`. This should succeed — customer service reps need policy details to help callers.

In [6]:
print("=" * 60)
print("ALICE (customer service) asks: 'Look up policy POL-10042'")
print("=" * 60)

response = alice_agent("Can you look up insurance policy POL-10042? I have a customer on the line asking about their coverage.")

print("\n" + "=" * 60)
print("RESULT: Alice (customer service) CAN look up policy details")
print("Cedar policy: PolicyLookup → ALLOW all authenticated users")
print("=" * 60)

ALICE (customer service) asks: 'Look up policy POL-10042'
Sure! Let me look that up right away.
Tool #1: PolicyLookup___lookup_policy
Here are the details for policy **POL-10042**:

- **Policy Holder:** Sarah Chen
- **Policy Type:** Auto Insurance
- **Status:** ✅ Active
- **Vehicle:** 2023 Toyota RAV4
- **Coverage:**
  - Liability: $500,000
  - Collision: $100,000
- **Monthly Premium:** $185.00
- **Effective Date:** June 1, 2025
- **Expiry Date:** June 1, 2026

The policy is currently active and valid through June 2026. Is there anything specific about Sarah's coverage you'd like to discuss or any other questions I can help with?
RESULT: Alice (customer service) CAN look up policy details
Cedar policy: PolicyLookup → ALLOW all authenticated users


## Cell 9: Alice Asks for Claims Data (BLOCKED by Gateway)

Alice asks for confidential claims information. The Cedar policy only permits Bob's `sub` to access ClaimsData.
The Gateway **enforces** this — Alice's agent doesn't even have `ClaimsData___query_claims` in its tool list.

**This is the key "whoa" moment — the Gateway enforces access control, not the LLM.**
A customer service rep should never see adjuster notes, fraud flags, or claim amounts.

In [7]:
print("=" * 60)
print("ALICE (customer service) asks: 'Show me the claims summary'")
print("=" * 60)

response = alice_agent("Can you show me the claims summary? I want to see all recent insurance claims and adjuster notes.")

print("\n" + "=" * 60)
print("RESULT: Alice CANNOT access claims data!")
print(f"Alice's available tools: {alice_tool_names}")
print("Gateway Cedar policy: ClaimsData not in Alice's permitted actions")
print("The Gateway doesn't even expose ClaimsData to Alice's agent")
print("=" * 60)

ALICE (customer service) asks: 'Show me the claims summary'
I appreciate you asking, but unfortunately **I don't have a tool available to look up claims summaries or adjuster notes**. My current capabilities are limited to looking up policy details by policy number.

To access claims information and adjuster notes, I'd suggest:

1. **Internal Claims System** – Log into your company's claims management platform directly for a full claims history and adjuster notes.
2. **Claims Department** – Reach out to your claims team who would have full access to that information.
3. **Supervisor/Admin Access** – If you need elevated access to claims data, your system administrator may be able to assist.

Is there anything else I can help with, such as looking up another policy? I'm happy to assist with what's available to me!
RESULT: Alice CANNOT access claims data!
Alice's available tools: ['PolicyLookup___lookup_policy']
Gateway Cedar policy: ClaimsData not in Alice's permitted actions
The Gatewa

## Cell 10: Bob Asks for Claims Data (ALLOWED by Gateway)

Now Bob asks the **exact same question**. His JWT `sub` matches the Cedar policy for ClaimsData.
The Gateway permits the call and returns the confidential claims information.

**Same question, different user, different result — enforced by Gateway Cedar policies, not the LLM.**

In [8]:
print("=" * 60)
print("BOB (claims adjuster) asks: 'Show me the claims summary'")
print("=" * 60)

response = bob_agent("Show me the claims summary. I need to review all recent insurance claims including adjuster notes and investigation status.")

print("\n" + "=" * 60)
print("RESULT: Bob (claims adjuster) CAN access claims data!")
print(f"Bob's available tools: {bob_tool_names}")
print("Same question as Alice. Same Gateway. Same endpoint.")
print("The ONLY difference: Bob's JWT sub matches the Cedar policy")
print("=" * 60)

BOB (claims adjuster) asks: 'Show me the claims summary'
I'll retrieve the claims summary for you right away!
Tool #1: ClaimsData___query_claims
Here is the **Claims Summary** for your review:

---

## 📊 Q1 2026 Quarterly Overview

| Metric | Value |
|---|---|
| Total Claims Filed | 147 |
| Total Claims Resolved | 112 |
| Total Amount Paid | $2.3M |
| Avg. Resolution Time | 18 days |
| Top Category | Auto - Collision (42%) |
| Fraud Flagged | 3 |

---

## 🗂️ Recent Claims Detail

### 1. CLM-2024-0891 — Sarah Chen
- **Type:** Auto - Collision
- **Policy:** POL-10042
- **Status:** ✅ Approved
- **Claimed:** $12,400 | **Approved:** $11,800
- **Filed:** 2025-12-15 | **Resolved:** 2026-01-20
- **Adjuster Notes:** Rear-end collision on Pacific Hwy. Liability confirmed via dashcam footage. Deductible of $600 applied.

---

### 2. CLM-2025-0023 — James Rodriguez
- **Type:** Home - Storm Damage
- **Policy:** POL-10078
- **Status:** 🔄 In Progress
- **Claimed:** $45,000 | **Approved:** Pending
- *

## Cell 11: Architecture Summary

In [9]:
print("""
==================================================================
    Open Insurance — AgentCore Gateway Demo Summary
==================================================================

  ARCHITECTURE:

    User (CLI)
      |
      | username + password (Resource Owner Password grant)
      v
    Okta Token Endpoint --> JWT (with groups, client_id claims)
      |
      v
    Strands Agent (JWT as Bearer token)
      |
      | MCP (StreamableHTTP + Bearer JWT)
      v
    AgentCore Gateway
      |-- Ingress Auth: JWT validation (signature, audience, client_id)
      |-- Cedar Policy Engine (ENFORCE mode)
      |     principal: AgentCore::OAuthUser::"<JWT sub>"
      |     action:    AgentCore::Action::"<TargetName>"
      |     resource:  AgentCore::Gateway::"<gateway-arn>"
      |
      +------------------+
      |                  |
      v                  v
    Lambda            Lambda
  (PolicyLookup)    (ClaimsData)

  AUTH FLOW:
    1. User authenticates via Okta ROPC grant -> JWT with sub claim
    2. Strands Agent attaches JWT as Bearer token on MCP connection
    3. Gateway validates JWT via Okta JWKS endpoint
    4. Cedar policies evaluated in ENFORCE mode
    5. Gateway permits or denies tool discovery and invocation
    6. No access control logic needed in the agent or MCP servers

  CEDAR POLICIES:
    PolicyLookup -> ALL AgentCore::OAuthUser -> ALLOW
    ClaimsData   -> AgentCore::OAuthUser::"bob@..." -> ALLOW
    ClaimsData   -> everyone else -> DENY (no matching permit)

  WHAT WE DEMONSTRATED:
    Alice (customer service)  -> Policy lookup  -> ALLOWED (Gateway permit)
    Alice (customer service)  -> Claims data    -> BLOCKED (Gateway deny)
    Bob (claims adjuster)     -> Claims data    -> ALLOWED (Gateway permit)

  KEY VALUE FOR INSURANCE:
    - ONE Gateway manages all internal insurance tools
    - Direct Okta OIDC -- no Cognito intermediary
    - Cedar policies ENFORCE access at the Gateway level
    - Customer service reps CANNOT see claims adjuster notes or fraud flags
    - Claims adjusters CAN see both policies and confidential claims
    - Agent needs zero access control logic
    - Zero auth code in MCP servers
    - Compliance team can audit Cedar policies directly

==================================================================
""")

# Cleanup MCPClients and httpx clients
alice_client.__exit__(None, None, None)
bob_client.__exit__(None, None, None)
print("Agents disconnected. Run cleanup in 01_setup.ipynb when done.")


    Open Insurance — AgentCore Gateway Demo Summary

  ARCHITECTURE:

    User (CLI)
      |
      | username + password (Resource Owner Password grant)
      v
    Okta Token Endpoint --> JWT (with groups, client_id claims)
      |
      v
    Strands Agent (JWT as Bearer token)
      |
      | MCP (StreamableHTTP + Bearer JWT)
      v
    AgentCore Gateway
      |-- Ingress Auth: JWT validation (signature, audience, client_id)
      |-- Cedar Policy Engine (ENFORCE mode)
      |     principal: AgentCore::OAuthUser::"<JWT sub>"
      |     action:    AgentCore::Action::"<TargetName>"
      |     resource:  AgentCore::Gateway::"<gateway-arn>"
      |
      +------------------+
      |                  |
      v                  v
    Lambda            Lambda
  (PolicyLookup)    (ClaimsData)

  AUTH FLOW:
    1. User authenticates via Okta ROPC grant -> JWT with sub claim
    2. Strands Agent attaches JWT as Bearer token on MCP connection
    3. Gateway validates JWT via Okta JWKS endp

---

## Cleanup

Removes the Web App, password-only auth policy, and reverts the Gateway's `allowedClients`.
This does **not** remove users, Cedar policies, Gateway, or Lambda targets (use `01_agentcore_setup.ipynb` cleanup for those).

In [ ]:
# --- Cleanup: revert Gateway, delete Web App and auth policy ---

import os
import json
import boto3
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

with open("gateway_config.json") as f:
    cfg = json.load(f)

OKTA_DOMAIN = os.environ["OKTA_DOMAIN"]
OKTA_API_TOKEN = os.environ["OKTA_API_TOKEN"]
OKTA_CLIENT_ID = os.environ.get("OKTA_CLIENT_ID", "")
OKTA_HEADERS = {"Authorization": f"SSWS {OKTA_API_TOKEN}", "Content-Type": "application/json"}

# 1. Revert Gateway allowedClients (remove Web App client ID)
agentcore = boto3.client("bedrock-agentcore-control", region_name=cfg["aws_region"])
gw = agentcore.get_gateway(gatewayIdentifier=cfg["gateway_id"])
current_clients = gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedClients"]

if OKTA_CLIENT_ID and OKTA_CLIENT_ID in current_clients:
    new_clients = [c for c in current_clients if c != OKTA_CLIENT_ID]
    agentcore.update_gateway(
        gatewayIdentifier=cfg["gateway_id"],
        name=gw["name"],
        roleArn=gw["roleArn"],
        protocolType=gw["protocolType"],
        authorizerType=gw["authorizerType"],
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": gw["authorizerConfiguration"]["customJWTAuthorizer"]["discoveryUrl"],
                "allowedAudience": gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedAudience"],
                "allowedClients": new_clients,
            }
        },
        policyEngineConfiguration=gw.get("policyEngineConfiguration", {}),
        exceptionLevel="DEBUG",
    )
    print(f"Reverted Gateway allowedClients to: {new_clients}")

# 2. Delete Okta Web App
APP_LABEL = "AgentCore Gateway Demo (Web)"
if OKTA_CLIENT_ID:
    apps_resp = requests.get(
        f"https://{OKTA_DOMAIN}/api/v1/apps?q={APP_LABEL}",
        headers=OKTA_HEADERS,
    )
    for app in apps_resp.json():
        if app.get("credentials", {}).get("oauthClient", {}).get("client_id") == OKTA_CLIENT_ID:
            requests.post(f"https://{OKTA_DOMAIN}/api/v1/apps/{app['id']}/lifecycle/deactivate", headers=OKTA_HEADERS)
            requests.delete(f"https://{OKTA_DOMAIN}/api/v1/apps/{app['id']}", headers=OKTA_HEADERS)
            print(f"Deleted Okta Web App: {app['label']} ({OKTA_CLIENT_ID})")
            break

# 3. Delete password-only auth policy
AUTH_POLICY_NAME = "Password Only (Demo)"
existing = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/policies?type=ACCESS_POLICY",
    headers=OKTA_HEADERS,
)
for p in existing.json():
    if p.get("name") == AUTH_POLICY_NAME:
        requests.delete(f"https://{OKTA_DOMAIN}/api/v1/policies/{p['id']}", headers=OKTA_HEADERS)
        print(f"Deleted auth policy: {AUTH_POLICY_NAME}")
        break

print("\n✅ Cleanup complete")